<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation, and observability trace extraction **without requiring any AWS access keys** from the recruiter. Credentials and secrets are retrieved securely via Cognito Identity Pools and AWS SSM.

In [41]:
# Install dependencies silently
!pip install boto3 requests -q
!sudo apt-get update -y -q > /dev/null && sudo apt-get install neofetch -y -q > /dev/null
!neofetch

import json
import time
import urllib.parse
import uuid

import boto3
import requests

# --- Global config / aliases ---
CFG = {
    "region": "us-east-1",
    "user_pool_id": "us-east-1_5pCxpIkx8",
    "client_id": "2r1ik1k110jbu6nfmuoegk5lns",
    "identity_pool_id": "us-east-1:c7680c24-fe96-4358-b305-6f43de1ca6c8",
    "agent_arn": "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-hvRgckAqaW",
    "fallback_user": "analyst_user",
    "fallback_password": "FinAIAgent2026@",
}

PARAMS = {
    "analyst_user": "/financial-ai/analyst-username",
    "analyst_password": "/financial-ai/analyst-password",
    "langfuse_pk": "/financial-ai/langfuse/public-key",
    "langfuse_sk": "/financial-ai/langfuse/secret-key",
}

AUTH_LOGIN_KEY = f"cognito-idp.{CFG['region']}.amazonaws.com/{CFG['user_pool_id']}"
ENCODED_ARN = urllib.parse.quote(CFG["agent_arn"], safe="")
AGENTCORE_URL = (
    f"https://bedrock-agentcore.{CFG['region']}.amazonaws.com"
    f"/runtimes/{ENCODED_ARN}/invocations"
)
MASTER_SESSION_ID = str(uuid.uuid4())

# Shared runtime state
STATE = {
    "username": "",
    "password": "",
    "access_token": None,
    "id_token": None,
}

# boto3 client aliases
idc = boto3.client("cognito-identity", region_name=CFG["region"])
idp = boto3.client("cognito-idp", region_name=CFG["region"])


def mk_ssm_client(creds: dict):
    """Create SSM client from temporary credentials."""
    return boto3.client(
        "ssm",
        region_name=CFG["region"],
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds.get("SecretAccessKey", creds.get("SecretKey")),
        aws_session_token=creds["SessionToken"],
    )


def get_guest_credentials():
    """Unauthenticated Cognito identity flow for guest credentials."""
    identity_id = idc.get_id(IdentityPoolId=CFG["identity_pool_id"])["IdentityId"]
    return idc.get_credentials_for_identity(IdentityId=identity_id)["Credentials"]


def get_authenticated_credentials(id_token: str):
    """Exchange Cognito IdToken for authenticated identity credentials."""
    identity_id = idc.get_id(
        IdentityPoolId=CFG["identity_pool_id"],
        Logins={AUTH_LOGIN_KEY: id_token},
    )["IdentityId"]
    return idc.get_credentials_for_identity(
        IdentityId=identity_id,
        Logins={AUTH_LOGIN_KEY: id_token},
    )["Credentials"]


def ssm_get(ssm_client, name: str) -> str:
    return ssm_client.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]


def query_financial_agent(prompt: str):
    if not STATE["access_token"]:
        print("Error: Access token is missing. Run authentication cell first.")
        return

    headers = {
        "Authorization": f"Bearer {STATE['access_token']}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": MASTER_SESSION_ID,
    }

    print(f"\n--- Query: {prompt} ---")
    resp = requests.post(
        AGENTCORE_URL,
        headers=headers,
        json={"prompt": prompt},
        stream=True,
        timeout=120,
    )
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {resp.text[:300]}")
        return

    for line in resp.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8")
        if decoded.startswith("data: "):
            data = json.loads(decoded[6:])
            if "event" in data:
                print(data["event"])
            elif "error" in data:
                print(f"Agent error: {data['error'][:300]}")


print(f"Session ID: {MASTER_SESSION_ID}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
            .-/+oossssoo+/-.
        `:+ssssssssssssssssss+:`
      -+ssssssssssssssssssyyssss+-
    .ossssssssssssssssssdMMMNysssso.
   /ssssssssssshdmmNNmmyNMMMMhssssss/
  +ssssssssshmydMMMMMMMNddddyssssssss+
 /sssssssshNMMMyhhyyyyhmNMMMNhssssssss/
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
 /sssssssshNMMMyhhyyyyhdNMMMNhssssssss/
  +sssssssssdmydMMMMMMMMddddyssssssss+
   /ssssssssssshdmNNNNmyNMMMMhssssss/
    .ossssssssssssssssssdMMMNysssso.
      -+sssssssssssssssssyyyssss+-
        `:+ssssssssssssssssss+:`
            .-/+oossssoo+/-.
root@d768b65d2734 
----------------- 
OS: Ubuntu 22.04.5 LTS x8

### 1. Retrieve Guest Login Credentials
We use the Cognito Identity Pool (Unauthenticated) to securely fetch the login credentials from AWS SSM.

In [ ]:
try:
    guest_creds = get_guest_credentials()
    ssm_guest = mk_ssm_client(guest_creds)

    STATE["username"] = ssm_get(ssm_guest, PARAMS["analyst_user"])
    STATE["password"] = ssm_get(ssm_guest, PARAMS["analyst_password"])
    print("Success: retrieved demo credentials via Guest Identity.")
except Exception as e:
    print(f"Warning: Guest retrieval failed ({str(e)[:120]}), using fallback.")
    STATE["username"] = CFG["fallback_user"]
    STATE["password"] = CFG["fallback_password"]


### 2. Authenticate with Amazon Cognito
Login using the credentials retrieved in the previous step.

In [43]:
if not STATE["username"] or not STATE["password"]:
    print("Error: Credentials not found. Run Step 1 successfully first.")
else:
    try:
        auth_response = idp.initiate_auth(
            ClientId=CFG["client_id"],
            AuthFlow="USER_PASSWORD_AUTH",
            AuthParameters={
                "USERNAME": STATE["username"],
                "PASSWORD": STATE["password"],
            },
        )
        STATE["access_token"] = auth_response["AuthenticationResult"]["AccessToken"]
        STATE["id_token"] = auth_response["AuthenticationResult"]["IdToken"]
        print("Success: Cognito authentication successful.")
    except Exception as e:
        print(f"Error: Authentication failed: {str(e)}")


Success: Cognito Authentication Successful.


### 3. Invoke the Agentcore Streaming Endpoint

In [ ]:
print(f"Agent URL configured: {AGENTCORE_URL}")
print(f"Session ID: {MASTER_SESSION_ID}")


### 4. Execute Required Financial Queries

In [45]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries: query_financial_agent(q)


--- Query: What is the stock price for Amazon right now? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What were the stock prices for Amazon in Q4 last year? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: Compare Amazon's recent stock performance to what analysts predicted in their reports. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: I’m researching AMZN give me the current price and any relevant information about their AI business. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What is the total amount of office space Amazon owned in North America in 2024? ---
Error 424: {"message":"An error occurred when s

### 5. Observability Verification (Authenticated AWS Identity)
Fetch Langfuse traces securely using temporary credentials obtained via the Cognito session.

**Session ID:** `e9ab4997-f01b-41ca-abde-bfb7cf06c258`

In [ ]:
try:
    if not STATE["id_token"]:
        raise ValueError("Missing IdToken. Run authentication cell first.")

    auth_creds = get_authenticated_credentials(STATE["id_token"])
    ssm_auth = mk_ssm_client(auth_creds)

    pk = ssm_get(ssm_auth, PARAMS["langfuse_pk"])
    sk = ssm_get(ssm_auth, PARAMS["langfuse_sk"])
    print(f"Success: retrieved Langfuse keys (PK: {pk[:7]}...)")

    if "placeholder" in sk.lower() or "00000000" in sk:
        print("NOTE: Langfuse keys are placeholders. Traces will not be available until real keys are stored in SSM.")
    else:
        print("Waiting 10s for trace propagation...")
        time.sleep(10)

        found = False
        for host in [
            "https://us.cloud.langfuse.com",
            "https://cloud.langfuse.com",
            "https://eu.cloud.langfuse.com",
        ]:
            trace_url = f"{host}/api/public/sessions/{MASTER_SESSION_ID}"
            trace_resp = requests.get(trace_url, auth=(pk, sk), timeout=30)
            if trace_resp.status_code == 200:
                print(f"\n--- Langfuse Traces (Host: {host}, Session: {MASTER_SESSION_ID}) ---")
                print(json.dumps(trace_resp.json(), indent=2)[:2000])
                found = True
                break

        if not found:
            print(f"No traces found for session {MASTER_SESSION_ID}. Check Langfuse dashboard manually.")
except Exception as e:
    print(f"Error: Observability retrieval failed: {str(e)}")
